In [27]:
"""
Self-Pruning Neural Network
"""

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm


class PrunableLinear(nn.Module):
    """
    A custom linear layer with learnable gates for self-pruning.

    Each weight has an associated gate parameter that controls whether
    the weight is active (gate ≈ 1) or pruned (gate ≈ 0).
    """

    def __init__(self, in_features, out_features):
        super(PrunableLinear, self).__init__()
        self.in_features = in_features
        self.out_features = out_features

        # Standard weight and bias parameters
        self.weight = nn.Parameter(torch.randn(out_features, in_features) * 0.01)
        self.bias = nn.Parameter(torch.zeros(out_features))

        # Gate scores - one per weight, initialized to encourage starting with all weights active
        self.gate_scores = nn.Parameter(torch.ones(out_features, in_features) * 2.0)

    def forward(self, x):
        """
        Forward pass with gated weights.

        Args:
            x: Input tensor of shape (batch_size, in_features)

        Returns:
            Output tensor of shape (batch_size, out_features)
        """
        gates = torch.sigmoid(self.gate_scores)

        pruned_weights = self.weight * gates

        # y = xW^T + b
        output = torch.nn.functional.linear(x, pruned_weights, self.bias)

        return output

    def get_gates(self):
        """Current gate values (after sigmoid)"""
        return torch.sigmoid(self.gate_scores)


class SelfPruningNetwork(nn.Module):
    """
    A feed-forward neural network with prunable layers for CIFAR-10 classification.
    """

    def __init__(self, input_size=3072, hidden_sizes=[512, 256, 128], num_classes=10):
        super(SelfPruningNetwork, self).__init__()

        self.flatten = nn.Flatten()

        layers = []
        prev_size = input_size

        for hidden_size in hidden_sizes:
            layers.append(PrunableLinear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            prev_size = hidden_size

        layers.append(PrunableLinear(prev_size, num_classes))

        self.network = nn.Sequential(*layers)
        self.prunable_layers = [layer for layer in self.network if isinstance(layer, PrunableLinear)]

    def forward(self, x):
        x = self.flatten(x)
        return self.network(x)

    def compute_sparsity_loss(self):
        """
        Compute L1 regularization on all gate values to encourage sparsity.

        Returns:
            Sum of all gate values (Scalar tensor)
        """
        sparsity_loss = 0.0
        for layer in self.prunable_layers:
            gates = layer.get_gates()
            sparsity_loss += gates.sum()

        return sparsity_loss

    def get_sparsity_stats(self, threshold=1e-2):
        """
        Calculate sparsity statistics for the network.

        Args:
            threshold: Gate values below this are considered pruned

        Returns:
            Dictionary with sparsity statistics
        """
        total_weights = 0
        pruned_weights = 0
        all_gates = []

        for layer in self.prunable_layers:
            gates = layer.get_gates().detach().cpu().numpy()
            all_gates.append(gates.flatten())

            total_weights += gates.size
            pruned_weights += (gates < threshold).sum()

        sparsity_percentage = (pruned_weights / total_weights) * 100
        all_gates = np.concatenate(all_gates)

        return {
            'total_weights': total_weights,
            'pruned_weights': pruned_weights,
            'active_weights': total_weights - pruned_weights,
            'sparsity_percentage': sparsity_percentage,
            'all_gates': all_gates
        }


def load_cifar10(batch_size=128):
    """Load CIFAR-10 dataset with standard preprocessing."""
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])

    trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                           download=True, transform=transform)
    trainloader = DataLoader(trainset, batch_size=batch_size,
                           shuffle=True, num_workers=2)

    testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                          download=True, transform=transform)
    testloader = DataLoader(testset, batch_size=batch_size,
                          shuffle=False, num_workers=2)

    return trainloader, testloader


def train_epoch(model, trainloader, optimizer, criterion, lambda_sparsity, device):
    """Train for one epoch."""
    model.train()
    running_loss = 0.0
    running_class_loss = 0.0
    running_sparsity_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in trainloader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs)

        # Classification loss
        class_loss = criterion(outputs, labels)

        # Sparsity loss
        sparsity_loss = model.compute_sparsity_loss()

        # Total loss
        total_loss = class_loss + lambda_sparsity * sparsity_loss

        # Backward pass
        total_loss.backward()
        optimizer.step()

        # Statistics
        running_loss += total_loss.item()
        running_class_loss += class_loss.item()
        running_sparsity_loss += sparsity_loss.item()

        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    return {
        'loss': running_loss / len(trainloader),
        'class_loss': running_class_loss / len(trainloader),
        'sparsity_loss': running_sparsity_loss / len(trainloader),
        'accuracy': 100. * correct / total
    }


def evaluate(model, testloader, criterion, device):
    """Evaluate the model on test set."""
    model.eval()
    test_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in testloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            test_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    return {
        'loss': test_loss / len(testloader),
        'accuracy': 100. * correct / total
    }


def train_model(lambda_sparsity, num_epochs=50, device='cuda'):
    """
    Train a self-pruning network with given sparsity parameter.

    Args:
        lambda_sparsity: Weight for sparsity regularization
        num_epochs: Number of training epochs
        device: Device to train on

    Returns:
        Trained model and training history
    """
    print(f"\n{'='*60}")
    print(f"Training with λ = {lambda_sparsity}")
    print(f"{'='*60}")

    trainloader, testloader = load_cifar10()

    model = SelfPruningNetwork().to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

    history = {
        'train_loss': [],
        'train_acc': [],
        'test_loss': [],
        'test_acc': [],
        'sparsity': []
    }

    best_acc = 0.0

    for epoch in range(num_epochs):
        train_stats = train_epoch(model, trainloader, optimizer, criterion,
                                  lambda_sparsity, device)

        test_stats = evaluate(model, testloader, criterion, device)

        sparsity_stats = model.get_sparsity_stats()

        history['train_loss'].append(train_stats['loss'])
        history['train_acc'].append(train_stats['accuracy'])
        history['test_loss'].append(test_stats['loss'])
        history['test_acc'].append(test_stats['accuracy'])
        history['sparsity'].append(sparsity_stats['sparsity_percentage'])

        scheduler.step()

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch+1}/{num_epochs}")
            print(f"  Train Loss: {train_stats['loss']:.4f} | Train Acc: {train_stats['accuracy']:.2f}%")
            print(f"  Test Loss: {test_stats['loss']:.4f} | Test Acc: {test_stats['accuracy']:.2f}%")
            print(f"  Sparsity: {sparsity_stats['sparsity_percentage']:.2f}%")

        if test_stats['accuracy'] > best_acc:
            best_acc = test_stats['accuracy']

    final_sparsity = model.get_sparsity_stats()
    print(f"\nFinal Results:")
    print(f"  Test Accuracy: {test_stats['accuracy']:.2f}%")
    print(f"  Sparsity Level: {final_sparsity['sparsity_percentage']:.2f}%")
    print(f"  Active Weights: {final_sparsity['active_weights']}/{final_sparsity['total_weights']}")

    return model, history, final_sparsity


def plot_gate_distribution(model, lambda_val, save_path='gate_distribution.png'):
    """Plot the distribution of gate values"""
    stats = model.get_sparsity_stats()
    gates = stats['all_gates']

    plt.figure(figsize=(10, 6))
    plt.hist(gates, bins=100, edgecolor='black', alpha=0.7)
    plt.xlabel('Gate Value', fontsize=12)
    plt.ylabel('Frequency', fontsize=12)
    plt.title(f'Distribution of Gate Values (λ = {lambda_val})', fontsize=14)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"Gate distribution plot saved to {save_path}")
    plt.close()


def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # random seed
    torch.manual_seed(42)
    np.random.seed(42)

    lambda_values = [0.0002, 0.002, 0.02]

    results = []

    for lambda_val in lambda_values:
        model, history, sparsity_stats = train_model(
            lambda_sparsity=lambda_val,
            num_epochs=50,
            device=device
        )

        results.append({
            'lambda': lambda_val,
            'test_accuracy': history['test_acc'][-1],
            'sparsity_percentage': sparsity_stats['sparsity_percentage'],
            'model': model,
            'history': history
        })

        # Plotting
        plot_gate_distribution(model, lambda_val,
                             f'gate_distribution_lambda_{lambda_val}.png')

    print(f"\n{'='*60}")
    print("SUMMARY OF RESULTS")
    print(f"{'='*60}")
    print(f"{'Lambda':<15} {'Test Accuracy':<20} {'Sparsity Level (%)':<20}")
    print(f"{'-'*60}")
    for result in results:
        print(f"{result['lambda']:<15} {result['test_accuracy']:<20.2f} {result['sparsity_percentage']:<20.2f}")

    with open('RESULTS.md', 'w') as f:

        f.write("## Results Summary\n\n")
        f.write("| Lambda | Test Accuracy (%) | Sparsity Level (%) |\n")
        f.write("|--------|-------------------|--------------------|\n")
        for result in results:
            f.write(f"| {result['lambda']} | {result['test_accuracy']:.2f} | ")
            f.write(f"{result['sparsity_percentage']:.2f} |\n")

        f.write("\n## Analysis\n\n")
        f.write("The results demonstrate the trade-off between model accuracy and sparsity:\n\n")
        f.write(f"- **Low λ ({lambda_values[0]})**: Minimal pruning, highest accuracy, ")
        f.write("network retains most connections.\n")
        f.write(f"- **Medium λ ({lambda_values[1]})**: Balanced trade-off, moderate pruning ")
        f.write("with acceptable accuracy loss.\n")
        f.write(f"- **High λ ({lambda_values[2]})**: Aggressive pruning, higher sparsity ")
        f.write("but potential accuracy degradation.\n\n")
        f.write("The gate distribution plots show the bimodal distribution characteristic of ")
        f.write("successful pruning, with many gates near 0 (pruned) and active gates clustered ")
        f.write("at higher values.\n")

    print("\nResults saved to RESULTS.md")
    print("Gate distribution plots saved as PNG files")


if __name__ == '__main__':
    main()

Using device: cuda

Training with λ = 0.0002
Epoch 1/50
  Train Loss: 299.7021 | Train Acc: 36.39%
  Test Loss: 1.5407 | Test Acc: 45.21%
  Sparsity: 0.00%
Epoch 5/50
  Train Loss: 168.8809 | Train Acc: 58.73%
  Test Loss: 1.3148 | Test Acc: 53.58%
  Sparsity: 0.00%
Epoch 10/50
  Train Loss: 51.8441 | Train Acc: 68.20%
  Test Loss: 1.2566 | Test Acc: 56.01%
  Sparsity: 0.00%
Epoch 15/50
  Train Loss: 20.8516 | Train Acc: 74.24%
  Test Loss: 1.2828 | Test Acc: 56.55%
  Sparsity: 0.00%
Epoch 20/50
  Train Loss: 10.8260 | Train Acc: 79.68%
  Test Loss: 1.3816 | Test Acc: 56.29%
  Sparsity: 0.00%
Epoch 25/50
  Train Loss: 8.0435 | Train Acc: 84.72%
  Test Loss: 1.4739 | Test Acc: 56.44%
  Sparsity: 0.00%
Epoch 30/50
  Train Loss: 6.3683 | Train Acc: 87.28%
  Test Loss: 1.6055 | Test Acc: 56.32%
  Sparsity: 43.94%
Epoch 35/50
  Train Loss: 5.2581 | Train Acc: 89.79%
  Test Loss: 1.7378 | Test Acc: 55.73%
  Sparsity: 57.58%
Epoch 40/50
  Train Loss: 4.4916 | Train Acc: 91.80%
  Test Loss: 1.